# 03 - Train Building Segmentation Baseline (U-Net ResNet34)

Kaggle notebook for training building segmentation model on xBD dataset.

In [ ]:
import os
import sys
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    # Attach xBD dataset + processed_xbd dataset to this notebook
    PROCESSED_DIR = Path('/kaggle/input/processed-xbd/processed_xbd')  # Adjust dataset slug
    OUTPUT_DIR = Path('/kaggle/working')
    sys.path.insert(0, '/kaggle/working')
    !pip install -q segmentation-models-pytorch albumentations
else:
    PROCESSED_DIR = Path('../data/processed/xbd')
    OUTPUT_DIR = Path('../outputs')
    sys.path.insert(0, str(Path('../src').resolve()))

print(f'Processed dir: {PROCESSED_DIR} (exists: {PROCESSED_DIR.exists()})')
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
import torch
from torch.utils.data import DataLoader
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

from rescuevision.data.datasets import XBDSegmentationDataset
from rescuevision.data.transforms import get_segmentation_transforms
from rescuevision.models.unet import create_unet
from rescuevision.models.losses import BCEDiceLoss
from rescuevision.training.train_segmentation import train_segmentation
from rescuevision.utils.seed import set_seed

## Configuration

In [ ]:
config = {
    "seed": 42,
    "device": "cuda",
    "dataset": {
        "processed_dir": str(PROCESSED_DIR),
        "split_csv": str(PROCESSED_DIR / "metadata" / "splits_event_aware.csv"),
        "mask_type": "building_binary",
        "image_size": 512,
        "batch_size": 8,
        "num_workers": 2,
    },
    "model": {
        "encoder": "resnet34",
        "encoder_weights": "imagenet",
        "in_channels": 3,
        "classes": 1,
    },
    "training": {
        "epochs": 50,
        "lr": 1e-4,
        "weight_decay": 1e-5,
        "scheduler": "cosine",
        "mixed_precision": True,
        "early_stopping_patience": 8,
        "save_best_metric": "iou",
        "gradient_accumulation_steps": 1,
    },
    "loss": {
        "bce_weight": 0.5,
        "dice_weight": 0.5,
    },
    "augmentation": {
        "horizontal_flip": 0.5,
        "vertical_flip": 0.5,
        "rotate90": 0.5,
        "brightness_contrast": 0.2,
        "normalize": True,
    },
}

set_seed(config["seed"])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')

## Create Datasets

In [ ]:
train_dataset = XBDSegmentationDataset(
    processed_dir=config["dataset"]["processed_dir"],
    split_csv=config["dataset"]["split_csv"],
    split="train",
    mask_type="building_binary",
    transform=get_segmentation_transforms(config["augmentation"], phase="train", image_size=512),
    image_size=512,
)

val_dataset = XBDSegmentationDataset(
    processed_dir=config["dataset"]["processed_dir"],
    split_csv=config["dataset"]["split_csv"],
    split="val",
    mask_type="building_binary",
    transform=get_segmentation_transforms(config["augmentation"], phase="val", image_size=512),
    image_size=512,
)

train_loader = DataLoader(train_dataset, batch_size=config["dataset"]["batch_size"],
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=config["dataset"]["batch_size"],
                        shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_dataset)} tiles, Val: {len(val_dataset)} tiles')

## Create Model and Train

In [ ]:
model = create_unet(
    encoder_name=config["model"]["encoder"],
    encoder_weights=config["model"]["encoder_weights"],
    in_channels=config["model"]["in_channels"],
    classes=config["model"]["classes"],
)

loss_fn = BCEDiceLoss(
    bce_weight=config["loss"]["bce_weight"],
    dice_weight=config["loss"]["dice_weight"],
)

output_dir = OUTPUT_DIR / "building_seg_unet"

best_metrics = train_segmentation(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    config=config,
    output_dir=str(output_dir),
    is_binary=True,
    num_classes=1,
)

print(f'\nBest metrics: {best_metrics}')

## Visualize Predictions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

model.eval()
model.to(device)

fig, axes = plt.subplots(3, 4, figsize=(20, 15))

for i in range(4):
    batch = next(iter(val_loader))
    images = batch["image"][:1].to(device)
    masks_gt = batch["mask"][:1].numpy()
    
    with torch.no_grad():
        pred = torch.sigmoid(model(images)).squeeze().cpu().numpy()
    pred_binary = (pred > 0.5).astype(np.uint8)
    
    # Denormalize image for display
    img = images[0].cpu().numpy().transpose(1, 2, 0)
    img = (img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))
    img = np.clip(img, 0, 1)
    
    axes[0, i].imshow(img)
    axes[0, i].set_title(f'Tile: {batch["tile_id"][0][:30]}')
    axes[1, i].imshow(masks_gt[0], cmap='gray')
    axes[1, i].set_title('Ground Truth')
    axes[2, i].imshow(pred_binary, cmap='gray')
    axes[2, i].set_title('Prediction')
    
    for ax in axes[:, i]:
        ax.axis('off')

plt.tight_layout()
plt.savefig(output_dir / 'predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## Save Checkpoint

In [ ]:
if IS_KAGGLE:
    print('Checkpoint saved at:', output_dir / 'checkpoints')
    print('Save this output as a Kaggle dataset for the inference notebook.')
    !ls -la {output_dir}/checkpoints/
else:
    print('Training complete. Checkpoint saved.')